# 🫀 퀘스트 46 · Q4-G — **예산 상한**과 **부담 추정의 재설계**

| | **MedKOS / `notebooks/quest46_q4g_budget_ceiling_burden.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② → 층① 전환점 |
| 부모 런 | `quest46_q4f_deployment_mode`(`20260805T0449`) |
| 성격 | **내 지표 오류를 고치고, 막힌 자리를 새 접근으로 뚫는다** |

## Q4-F 가 남긴 것

**확정** — `J2 ✅ 0.00e+00`. 예산 방식에서 `raw` ≡ `A_em` 이 **정확히** 성립한다.
**처방에서 층②(사전확률 보정·부담 특징·π̂)를 통째로 뺄 수 있다.**

**그런데 문제 셋이 드러났다.**

### ① 예산 상한 — 내 J5 가 두 종류의 실패를 섞었다

```
레코드   유병률   S총수  예산k   상한    실측    달성률   AUROC
#48     0.5764   1818    190   0.104  0.0231   22.1%  0.7681   ← 예산이 S 보다 작다
#64     0.2781    628    136   0.216  0.0191    8.8%  0.7762   ← 같은 이유
#38     0.0895    227    153   0.673  0.0088    1.3%  0.8479   ← **진짜 모델 실패**
#61     0.0335     64    115   1.000  0.0312    3.1%  0.7679   ← **진짜 모델 실패**
```

`#48`·`#64` 는 **모델이 완벽해도** 상한이 0.104·0.216 이다. `#38`·`#61` 은 상한이
0.673·1.000 인데 1.3%·3.1% 밖에 못 낸다. **완전히 다른 실패인데 한 표에 섞어**
「민감도~유병률 상관 −0.5697」로 보고했다. K1 이 **상한과 달성률**로 갈라 다시 읽는다.

### ② AUROC 가 이 국면의 잘못된 요약이다

`#38` 은 **AUROC 0.8479** 인데 상위 6% 로 양성의 **0.9%** 만 잡는다. AUROC 는 **전체
순위**를 보고, 예산 국면은 **최상위 꼬리**만 쓴다. K1 이 **recall@k** 를 주 진단으로
올리고 AUROC 와의 괴리를 명시한다.

### ③ 고정 예산으로는 82배 유병률 범위를 못 덮는다

```
기록당 300개 예산   유병률   민감도    PPV    오경보
저유병률 사분위     0.0150  0.8752  0.1086   89.1%   ← 300개 중 33개만 진짜
고유병률 사분위     0.2175  0.4600  0.7017   29.8%   ← 절반 넘게 놓친다
```

**예산이 부담에 비례해야** 한다(`k_r ∝ π_r`). 그런데—

### ④ ★★★ π̂ 가 무정보다 — 순환이 걸렸다

```
em     MAE 0.1790  ρ(π̂,π*) −0.0019
mean_p MAE 0.0724  ρ         −0.0684
bbse   MAE 0.1709  ρ         −0.0147
count  MAE 0.0885  ρ         −0.0503
```

**네 추정기 전부 상관이 0 또는 음수다.** 「부정확」이 아니라 **무정보**다. 레코드 48
(π\* 0.5764)에서 네 개 모두 0.00~0.04 로 **15~58배 과소추정**한다.

**왜인가** — 네 추정기가 **전부 박동별 보정 확률의 함수**다. 보정은 유병률이 낮은 DEV
레코드에서 적합되므로 **확률 눈금 자체가 낮은 쪽에 고정**된다. 고부담 기록에 필요한
p ≈ 0.5 를 모델이 낼 수 없다.

**⇒ 접근을 바꾼다.** 박동을 세지 말고 **기록 수준 요약통계에서 π 를 직접 회귀**한다.
RR 분포의 형태(짧은 RR 비율·분위수·이봉성·변동성)는 부담과 직접 연결되고, **보정 눈금을
거치지 않는다.**

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **K0** | 코호트 · Platt 기울기 | 구성. 깨지면 **중단** |
| **K1 ★★★ 눈금 정정** | 예산 **상한**과 **달성률**, recall@k vs AUROC 괴리 | 관문 아님. Q4-F J5 정정 |
| **K2 ★★★ 주 관문** | **기록 수준 회귀** π̂ 가 기존 넷을 이기는가 (ρ · MAE) | 측정된 영점 상단 초과 |
| **K3 ★★** | **적응 예산**(k ∝ π̂) vs 균등 예산 — 같은 총 예산 | 측정된 영점 상단 초과 |
| **K4** | **오라클 예산**(k ∝ π\*) = 적응 예산의 상한 | 관문 아님(R36 ①) |
| **K5** | 저유병률 PPV · 고유병률 민감도를 **따로** 보고 | 관문 아님 |
| **K6** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **K2 ✅ · K3 ✅** → **정량 트랙이 열리고 적응 예산으로 민감도가 오른다**
- **K2 ✅ · K3 미결** → 부담은 읽히는데 예산 배분으로 전환이 안 된다
- **K2 ❌** → 기록 수준으로도 부담을 못 읽는다 → **정량 트랙을 접는다**(상한 보고)

⚠️ **새 데이터 0** — `svdb_data5.npz` 만. 회귀 특징은 기존 RR 에서 파생한다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def boot_rho(x, y, seed, nb=3000, q=2.5):
    """★ 상관의 CI — 레코드 군집 부트스트랩."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 4:
        return float("nan"), float("nan"), float("nan"), len(x)
    rng = np.random.RandomState(seed); v = []
    for _ in range(nb):
        j = rng.randint(0, len(x), len(x))
        if np.std(x[j]) > 1e-12 and np.std(y[j]) > 1e-12:
            v.append(float(np.corrcoef(x[j], y[j])[0, 1]))
    return (float(np.corrcoef(x, y)[0, 1]), float(np.percentile(v, q)),
            float(np.percentile(v, 100 - q)), len(x))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수
DEV_EVERY = 4
FLAG_K = (100, 300)
MAIN_K = 300               # 주 지표 예산(기록당)
RIDGE_A = 1.0              # 기록 수준 회귀의 정규화(사전 고정)
MAX_NEG_SLOPE = 0.10

NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 10
N_PERM_REG = 20 if SMOKE else 200     # ★ 회귀 영점은 ridge 재적합뿐이라 싸다

ARM = "raw"                # Q4-F J2 로 층② 를 뺐다
PI_EST = ("em", "mean_p", "bbse", "count", "reg")     # ★ `reg` 가 이 런의 새 팔
NEW_EST = "reg"
BASE_EST = ("em", "mean_p", "bbse", "count")
READ_ORDER = ("K0", "K1", "K2", "K3", "K4", "K5", "K6")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(   # Q4-F(`20260805T0449`) 실측 — 재현 앵커
    n_ok=56, n_beat=138898, mean_prev=0.0837, dom_rec=48,
    macro_ap=0.5796, macro_auc=0.9420,
    j4=dict(em=(0.1790, -0.0019), mean_p=(0.0724, -0.0684),
            bbse=(0.1709, -0.0147), count=(0.0885, -0.0503)),
    j4_dom48=dict(em=0.0100, mean_p=0.0360, bbse=0.0000, count=0.0101, true=0.5764),
    j3=dict(k100=(0.4548, 0.4639), k300=(0.7459, 0.3620)),
    j3band=[(0.0150, 0.8752, 0.1086), (0.0326, 0.8745, 0.2124),
            (0.0698, 0.7739, 0.4255), (0.2175, 0.4600, 0.7017)],
    j5_worst=[("#38", 0.0895, 0.0088, 0.8479, 227), ("#61", 0.0335, 0.0312, 0.7679, 64),
              ("#48", 0.5764, 0.0231, 0.7681, 1818), ("#64", 0.2781, 0.0191, 0.7762, 628)])

RULE_CHECK = {
    "R11 매크로":       "환자 단위. 지표는 **상한과 함께** 읽는다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "★★ 회귀·문턱을 **그 fold 의 학습 레코드에서만** 적합(LORO)",
    "R26 / R38 ②":      "대비의 **영점**을 측정. 못 쟀으면 **안 읽는다**",
    "R29 ② 분기 금지":   "K0 가 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ② 문턱 금지":  "예산 `FLAG_K`·정규화 `RIDGE_A` 를 **사전 고정**",
    "R35 ① 자 먼저":    "★★★ **K1 이 자다** — 상한 없이는 민감도를 못 읽는다",
    "R36 ① 상한":       "★ K4 오라클 예산은 **상한이지 방법이 아니다**",
    "R40 ① λ ≠ 타당성":  "★★★ **AUROC 가 좋아도 상위 꼬리가 좋다는 보장은 없다** — K1 이 실측",
    "R40 ② 같은 통계":  "필요표본을 **관문 문턱 기준**으로",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4g_budget_ceiling_burden", quest="ailab-2026-0046",
    step="budget-ceiling-and-burden", parent_exp=["quest46_q4f_deployment_mode"],
    purpose=("**내 지표 오류를 고치고, 막힌 자리를 새 접근으로 뚫는다.** Q4-F 가 처방을 "
             "확정했지만(J2 ✅ — 예산 방식에서 `raw` ≡ `A_em` 이 0.0e+00 이라 **층② 를 "
             "통째로 뺄 수 있다**) 문제 셋을 드러냈다. ① **예산 상한을 안 봤다** — 레코드 48 은 "
             "S 가 1818 개인데 예산이 190 개라 **최대 가능 민감도가 0.104** 다. 그런데 J5 는 "
             "이걸 `#38`(상한 0.673 인데 0.0088 = 진짜 모델 실패)과 **한 표에 섞어** "
             "「민감도~유병률 상관 −0.5697」로 보고했다. ② **AUROC 가 이 국면의 잘못된 "
             "요약**이다 — `#38` 은 AUROC **0.8479** 인데 상위 6% 로 양성의 **0.9%** 만 잡는다. "
             "AUROC 는 전체 순위를 보고 예산 국면은 **최상위 꼬리**만 쓴다. ③ **고정 예산으로 "
             "82배 유병률 범위를 못 덮는다** — 기록당 300개에서 저유병률은 민감도 0.875/"
             "**오경보 89%**, 고유병률은 PPV 0.702/**민감도 0.460** 이다. 예산이 부담에 "
             "비례해야 하는데 ④ **π̂ 가 무정보다** — 네 추정기 전부 ρ(π̂,π\\*) 가 0 또는 "
             "음수(−0.0019 ~ −0.0684)이고 레코드 48 에서 15~58배 과소추정한다. "
             "**왜인가**: 넷 다 **박동별 보정 확률의 함수**이고, 보정은 저유병률 DEV 에서 "
             "적합되므로 **확률 눈금이 낮은 쪽에 고정**된다. ⇒ 이 런은 박동을 세지 말고 "
             "**기록 수준 요약통계(RR 분포의 형태)에서 π 를 직접 회귀**한다 — 보정 눈금을 "
             "거치지 않는 경로다."),
    dataset="SVDB — svdb_data5.npz (새 데이터 0 · 회귀 특징은 기존 RR 파생)",
    arm=ARM, pi_est=list(PI_EST), new_est=NEW_EST, flag_k=list(FLAG_K), main_k=MAIN_K,
    ridge_alpha=RIDGE_A, read_order=READ_ORDER,
    dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM, n_perm_reg=N_PERM_REG, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "K0": "코호트 + Platt 기울기. 깨지면 **중단**",
        "K1": "★★★ **자(관문 아님)** — 레코드별 **예산 상한** min(1, k/S) 과 **달성률** "
              "sens/상한 을 낸다. 그리고 **recall@k** 를 AUROC·PR-AUC 와 나란히 놓아 "
              "**AUROC 가 상위 꼬리를 대변하지 못함**을 실측한다(Q4-F `#38` AUROC 0.8479 · "
              "상위 6% 로 0.9%). Q4-F J5 의 정정이다",
        "K2": "★★★ **주 관문 — 기록 수준 부담 회귀.** RR 분포의 형태(짧은 RR 비율·분위수·"
              "변동성·이봉성 대리)에서 logit(π) 를 **ridge 로 직접 회귀**한다(LORO). "
              "기존 네 추정기가 ρ ≈ 0 인 이유는 **전부 박동별 보정 확률의 함수**여서 "
              "확률 눈금에 갇히기 때문이다. 회귀는 그 경로를 우회한다. "
              "판정 = ρ(π̂,π\\*) 가 **측정된 영점 상단**을 넘는가",
        "K3": "★★ **적응 예산** — 같은 총 예산을 `k_r ∝ π̂_r` 로 배분한 뒤 균등 배분과 "
              "레코드별 민감도를 비교한다. K2 가 서야 의미가 있다",
        "K4": "**오라클 예산**(`k_r ∝ π\\*_r`) — 적응 예산이 아무리 잘해도 여기까지라는 "
              "선(R36 ①). **상한이지 방법이 아니다**",
        "K5": "저유병률의 **PPV 문제**와 고유병률의 **민감도 문제**를 따로 보고 — 하나의 "
              "예산으로 둘 다 못 만족한다는 게 요점이다",
        "K6": "결론 검산표"},
    caveat=("★★★ **K1 이 이 런의 자다** — 상한을 안 보면 「민감도 0.02」가 모델 실패인지 "
            "예산 부족인지 구분이 안 된다. Q4-F 가 정확히 그 오류를 냈다. "
            "★★ **AUROC 를 이 국면의 요약으로 쓰지 않는다**(R40 ①) — 예산 국면은 상위 "
            "꼬리만 쓰므로 **recall@k** 가 맞는 지표다. Q4-E 의 「AUROC 0.9418」은 여전히 "
            "참이지만 **배포 성능을 대변하지 않는다**. "
            "★ **K2 가 실패하면 정량 트랙을 접는다** — 그때는 상한(오라클 예산 K4)만 남기고, "
            "고부담 환자는 **탐지·정량 둘 다 이 모델로는 안 된다**고 기록한다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4g_budget_ceiling_burden", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-G — 예산 상한과 부담 추정의 재설계**")
run.log("  ★★★ K1 — **상한·달성률·recall@k** 로 Q4-F J5 를 정정한다")
run.log("  ★★★ K2(주 관문) — **기록 수준 회귀**로 π 를 직접 읽는다(보정 눈금 우회)")
run.log(f"  ★★ K3 — 적응 예산 `k ∝ π̂` · K4 — 오라클 예산 `k ∝ π*`(상한)")
run.log(f"  ▸ Q4-F 앵커 — π̂ 네 추정기 ρ 전부 0 또는 음수"
        f"({[REF['j4'][e][1] for e in BASE_EST]})")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【K-0】 코호트 · LORO · ★ 기록 수준 특징
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【K-0】 코호트 · LORO · 기록 수준 특징")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
RHY = np.nan_to_num(np.c_[_med - pre,
                          np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                          post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS = {int(r): np.where(RID == r)[0] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NS_ = {r: int(TT_[IDXS[r]].sum()) for r in REC_OK}
NN_ = {r: int(len(IDXS[r])) for r in REC_OK}
NRE = len(REC_OK); NB_TOT = int(sum(NN_.values()))
MEAN_PREV = float(np.mean([BURD[r] for r in REC_OK]))
DOM_REC = int(max(REC_OK, key=lambda r: NS_[r]))
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 총 박동 {NB_TOT} · 평균 유병률 {MEAN_PREV:.4f}")

# ── ★★★ 기록 수준 특징 — **박동별 확률을 안 거친다**. 그게 이 런의 요점이다.
#    기존 네 추정기는 전부 보정 확률의 함수라 **확률 눈금에 갇힌다**(Q4-F ρ ≈ 0).
def rec_feats(r):
    idx = IDXS[r]; p = pre[idx]; q = post[idx]
    med = float(np.median(p)); rel = p / (med + 1e-9)
    f = [float(np.mean(rel < t)) for t in (0.70, 0.80, 0.85, 0.90, 0.95)]   # 짧은 RR 비율
    f += [float(np.quantile(rel, t)) for t in (0.05, 0.10, 0.25, 0.50)]     # 하위 분위수
    f += [float(np.std(rel)), float(np.mean(np.abs(np.diff(p)) / (med + 1e-9)))]
    f += [float(np.mean((q / (med + 1e-9)) > 1.10)),                        # 보상성 휴지기
          float(np.corrcoef(p[:-1], q[:-1])[0, 1]) if len(p) > 3 else 0.0]
    # 이봉성 대리 — 상대 RR 히스토그램의 두 봉우리 사이 골
    h, _ = np.histogram(np.clip(rel, 0.4, 1.6), bins=24, range=(0.4, 1.6))
    h = h / max(1, h.sum()); k_ = int(np.argmax(h))
    f += [float(h.max()), float(h[:max(1, k_)].min() if k_ > 0 else 0.0),
          float(np.sum(h[:12]))]
    return np.nan_to_num(np.array(f, float), nan=0.0, posinf=0.0, neginf=0.0)

RF = {r: rec_feats(r) for r in REC_OK}
NFEAT = len(RF[REC_OK[0]])
run.log(f"  ★★★ **기록 수준 특징 {NFEAT}개** — 짧은 RR 비율 · 상대 RR 분위수 · 변동성 · "
        f"보상성 휴지기 · 이봉성 대리")
run.log(f"     **박동별 보정 확률을 안 거친다** — 기존 네 추정기가 갇힌 그 눈금을 우회한다")

EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

SLOPES = []
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    return (lambda v: 1.0 / (1.0 + np.exp(-(a * np.asarray(v, float) + b))),
            lambda v: a * np.asarray(v, float) + b)

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def loro(y_override=None):
    out = np.full(len(K), np.nan); prob = np.full(len(K), np.nan); dev = {}; reg = {}
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        Ftr = RHY[tr]; fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii: lr.decision_function((RHY[ii] - fmu) / fsd)
        cp, cl = make_cal(sc(dv), TT_[dv])
        out[te] = cl(sc(te)); prob[te] = cp(sc(te))
        dev[held] = dict(logit=cl(sc(dv)), y=TT_[dv].astype(int), pi_tr=float(TT_[dv].mean()))
        # ★★★ 기록 수준 회귀 — **held-out 을 뺀 레코드에서만** 적합(R22)
        fit_r = tr_r + dv_r
        X = np.array([RF[r] for r in fit_r], float)
        xm, xs = X.mean(0), X.std(0) + 1e-9
        yv = np.array([logit(BURD[r]) for r in fit_r], float) if y_override is None else \
             np.array([logit(float(np.mean(y_override[r]))) if r in y_override else
                       logit(BURD[r]) for r in fit_r], float)
        rg = Ridge(alpha=RIDGE_A).fit((X - xm) / xs, yv)
        reg[held] = float(1.0 / (1.0 + np.exp(-float(
            rg.predict(((RF[held] - xm) / xs).reshape(1, -1))[0]))))
    return out, prob, dev, reg

run.log("  LORO · 회귀 골격 완료 — 회귀는 **held-out 을 뺀 레코드에서만** 적합한다(R22)")
CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=NRE, n_beat=NB_TOT, mean_prev=MEAN_PREV,
                        dom_rec=DOM_REC, n_feat=NFEAT)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【K-A】 실행 · K0 · ★★★ K1 상한·달성률·recall@k
run.log("\n" + "=" * 100)
run.log("【K-A】 실행 · K0 · ★★★ K1 — **상한 없이는 민감도를 못 읽는다**(Q4-F J5 정정)")
run.log("=" * 100)
T0 = time.time()
L, P, DEV, REG = loro()
run.log(f"  ({time.time()-T0:.0f}초) LORO 완료")

sl = np.array(SLOPES, float); neg = int((sl <= 0).sum())
if np.median(sl) <= 0 or neg / max(1, len(sl)) > MAX_NEG_SLOPE:
    raise AssetError(f"K0 실패 — 기울기 중앙 {np.median(sl):.4f} · 음수 {neg}(R29 ②)")
g_("K0", "✅ 지지", f"Platt 기울기 {len(sl)}개 · 중앙 {np.median(sl):+.4f} · 음수 {neg}")

AP = {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[IDXS[r]])) for r in REC_OK}
AUC = {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[IDXS[r]])) for r in REC_OK}

def recall_at(r, k):
    idx = IDXS[r]; sc = L[idx]; yy = TT_[idx]
    k = int(min(max(1, k), len(idx)))
    fl = sc >= np.partition(sc, -k)[-k]
    tp = int((fl & yy).sum())
    return dict(sens=tp / max(1, int(yy.sum())), ppv=tp / max(1, int(fl.sum())),
                tp=tp, fn=int(yy.sum()) - tp, flagged=int(fl.sum()),
                ceil=float(min(1.0, k / max(1, int(yy.sum())))))

# ── ★★★ K1 — 상한과 달성률
run.log(f"\n  ★★★ K1 — 예산 {MAIN_K}개/기록에서의 **상한 min(1, k/S)** 과 **달성률**")
R1 = {r: recall_at(r, MAIN_K) for r in REC_OK}
for r in REC_OK:
    R1[r]["ach"] = R1[r]["sens"] / R1[r]["ceil"] if R1[r]["ceil"] > 0 else float("nan")
capped = [r for r in REC_OK if R1[r]["ceil"] < 0.999]
run.log(f"    **예산이 S 보다 작아 상한에 걸린 레코드 {len(capped)}/{NRE}** — 이들은 "
        f"모델이 완벽해도 민감도가 상한을 못 넘는다")
run.log(f"    상한 중앙 {np.median([R1[r]['ceil'] for r in REC_OK]):.4f} · "
        f"최소 {min(R1[r]['ceil'] for r in REC_OK):.4f}")
run.log(f"    민감도 평균 {np.mean([R1[r]['sens'] for r in REC_OK]):.4f} · "
        f"**달성률 평균 {np.mean([R1[r]['ach'] for r in REC_OK]):.4f}**")

run.log(f"\n    Q4-F J5 의 「최악 5」를 **상한과 함께** 다시 읽는다")
run.log(f"    {'레코드':<8}{'유병률':>9}{'S총수':>7}{'상한':>8}{'민감도':>9}{'달성률':>9}"
        f"{'AUROC':>8}{'lift':>8}{'진단':>10}")
for nm, pv_, sv_, au_, S_ in REF["j5_worst"]:
    r = int(nm[1:])
    if r not in R1:
        continue
    tag = "예산 부족" if R1[r]["ceil"] < 0.5 else "**모델 실패**"
    run.log(f"    {nm:<8}{BURD[r]:>9.4f}{NS_[r]:>7d}{R1[r]['ceil']:>8.3f}"
            f"{R1[r]['sens']:>9.4f}{R1[r]['ach']:>9.1%}{AUC[r]:>8.4f}"
            f"{AP[r]/BURD[r]:>7.1f}배{tag:>12}")
run.log(f"    ⇒ Q4-F 는 이 둘을 **한 표에 섞어** 「민감도~유병률 상관 −0.5697」로 읽었다")

# ── ★★★ AUROC 가 상위 꼬리를 대변하지 못한다
run.log(f"\n  ★★★ **AUROC 는 이 국면의 요약이 아니다**(R40 ①)")
au = np.array([AUC[r] for r in REC_OK]); ac = np.array([R1[r]["ach"] for r in REC_OK])
lf = np.array([AP[r] / BURD[r] for r in REC_OK])
run.log(f"    ρ(AUROC, 달성률) = **{np.corrcoef(au, ac)[0,1]:+.4f}** · "
        f"ρ(lift, 달성률) = **{np.corrcoef(lf, ac)[0,1]:+.4f}**")
hi_auc_lo_ach = [r for r in REC_OK if AUC[r] > 0.85 and R1[r]["ach"] < 0.5]
run.log(f"    AUROC > 0.85 인데 달성률 < 50% 인 레코드 **{len(hi_auc_lo_ach)}/{NRE}** — "
        + (", ".join(f"#{r}(AUROC {AUC[r]:.3f} · 달성 {R1[r]['ach']:.0%})"
                     for r in sorted(hi_auc_lo_ach, key=lambda x: -AUC[x])[:5])
           if hi_auc_lo_ach else "없다"))
g_("K1", "(관문 아님)",
   f"예산 {MAIN_K}개에서 **상한에 걸린 레코드 {len(capped)}/{NRE}** · 달성률 평균 "
   f"{np.mean(ac):.4f} · ρ(AUROC, 달성률) {np.corrcoef(au, ac)[0,1]:+.4f} — "
   f"**AUROC 는 상위 꼬리를 대변하지 못한다**")
CONFIG["K0"] = dict(slope_med=float(np.median(sl)), n_neg=neg, n=len(sl))
CONFIG["K1"] = dict(n_capped=len(capped), ceil_med=float(np.median([R1[r]["ceil"] for r in REC_OK])),
                    sens_mean=float(np.mean([R1[r]["sens"] for r in REC_OK])),
                    ach_mean=float(np.mean(ac)),
                    rho_auc_ach=float(np.corrcoef(au, ac)[0, 1]),
                    rho_lift_ach=float(np.corrcoef(lf, ac)[0, 1]),
                    n_hi_auc_lo_ach=len(hi_auc_lo_ach),
                    per_rec={str(r): dict(prev=BURD[r], nS=NS_[r], ceil=R1[r]["ceil"],
                                          sens=R1[r]["sens"], ach=R1[r]["ach"],
                                          auc=AUC[r], ap=AP[r]) for r in REC_OK})
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【K-B】 ★★★ K2 주 관문 — 기록 수준 부담 회귀
run.log("\n" + "=" * 100)
run.log("【K-B】 ★★★ K2 — **기록 수준 회귀**가 박동별 추정기를 이기는가")
run.log("=" * 100)
run.log("  기존 넷은 **전부 박동별 보정 확률의 함수**다 — 보정이 저유병률 DEV 에서")
run.log("  적합되므로 **확률 눈금이 낮은 쪽에 고정**된다(Q4-F ρ ≈ 0 · 레코드 48 에서 15~58배")
run.log("  과소추정). 회귀는 **그 경로를 거치지 않는다**.")

PIH = {e: {} for e in PI_EST}
for r in REC_OK:
    d = DEV[r]; te = IDXS[r]; p_te = P[te]; pi_tr = d["pi_tr"]
    PIH["mean_p"][r] = float(np.mean(p_te))
    PIH["em"][r] = float(em_prior(p_te, pi_tr))
    thr = float(np.quantile(d["logit"], 0.95))
    rate = float((L[te] >= thr).mean()); PIH["count"][r] = rate
    dy = d["y"].astype(bool); dl = d["logit"]
    tpr = float((dl[dy] >= thr).mean()) if dy.any() else np.nan
    fpr = float((dl[~dy] >= thr).mean()) if (~dy).any() else np.nan
    PIH["bbse"][r] = (float(np.clip((rate - fpr) / (tpr - fpr), 0.0, 1.0))
                      if np.isfinite(tpr) and np.isfinite(fpr) and abs(tpr - fpr) > 1e-6
                      else np.nan)
    PIH["reg"][r] = REG[r]

TRUE = np.array([BURD[r] for r in REC_OK], float)
run.log(f"\n  {'추정기':<10}{'평균|π̂−π*|':>14}{'중앙':>9}{'상대중앙':>11}"
        f"{'ρ(π̂,π*)':>26}")
K2 = {}
for e in PI_EST:
    v = np.array([PIH[e][r] for r in REC_OK], float); m = np.isfinite(v)
    ae = np.abs(v[m] - TRUE[m])
    rho, rlo, rhi, rn = boot_rho(v[m], TRUE[m], SEED0 + 31 + len(e), NB_BOOT)
    K2[e] = dict(mae=float(ae.mean()), med=float(np.median(ae)),
                 rel_med=float(np.median(ae / TRUE[m])), rho=rho, rho_lo=rlo, rho_hi=rhi,
                 n=int(m.sum()))
    star = " ★" if e == NEW_EST else ""
    run.log(f"  {e:<10}{K2[e]['mae']:>14.4f}{K2[e]['med']:>9.4f}{K2[e]['rel_med']:>10.1%}"
            f"{rho:>+11.4f} [{rlo:+.4f},{rhi:+.4f}]{star}")
run.log(f"  (Q4-F 앵커 — " + " · ".join(f"{e} MAE {REF['j4'][e][0]}/ρ {REF['j4'][e][1]}"
                                        for e in BASE_EST) + ")")
run.log(f"\n  ★ 지배 레코드 {DOM_REC}(π* {BURD[DOM_REC]:.4f}) — "
        + " · ".join(f"{e} {PIH[e][DOM_REC]:.4f}" for e in PI_EST))

# ── ★★ 영점 — 학습 라벨을 치환하면 회귀의 ρ 가 얼마인가
run.log(f"\n  ★★ **영점** — 학습 레코드의 π 를 치환한 뒤 같은 회귀 (reps={N_PERM_REG})")
nul = []
for s_ in range(N_PERM_REG):
    rr = np.random.RandomState(SEED0 + 500 + s_)
    est = {}
    for held in REC_OK:
        fit_r = [r for r in REC_OK if r != held]
        perm = rr.permutation(len(fit_r))
        X = np.array([RF[r] for r in fit_r], float)
        xm, xs = X.mean(0), X.std(0) + 1e-9
        yv = np.array([logit(BURD[fit_r[perm[i]]]) for i in range(len(fit_r))], float)
        rg = Ridge(alpha=RIDGE_A).fit((X - xm) / xs, yv)
        est[held] = float(1.0 / (1.0 + np.exp(-float(
            rg.predict(((RF[held] - xm) / xs).reshape(1, -1))[0]))))
    v = np.array([est[r] for r in REC_OK], float)
    nul.append(float(np.corrcoef(v, TRUE)[0, 1]))
NR = boot_mean(nul, SEED0 + 61, NB_BOOT)
run.log(f"    ρ 의 영점 **{NR[0]:+.4f}** [{NR[1]:+.4f}, {NR[2]:+.4f}] (reps {N_PERM_REG})")
K2_THR = max(0.0, NR[2]) if np.isfinite(NR[2]) else float("nan")
run.log(f"    ▸ 문턱 = max(0, 영점 상단) **{K2_THR:+.4f}**")
k2v = decide(K2[NEW_EST]["rho_lo"], K2[NEW_EST]["rho_hi"], K2_THR, ">")
best_base = max(BASE_EST, key=lambda e: K2[e]["rho"])
g_("K2", k2v,
   f"**기록 수준 회귀** ρ {K2[NEW_EST]['rho']:+.4f} [{K2[NEW_EST]['rho_lo']:+.4f}, "
   f"{K2[NEW_EST]['rho_hi']:+.4f}] · MAE {K2[NEW_EST]['mae']:.4f} vs 기존 최선 "
   f"`{best_base}` ρ {K2[best_base]['rho']:+.4f}/MAE {K2[best_base]['mae']:.4f} · "
   f"문턱 {K2_THR:+.4f} — "
   + ("**부담이 기록 수준에서 읽힌다 → 정량 트랙이 열린다**" if k2v.startswith("✅") else
      ("기록 수준으로도 못 읽는다 — **정량 트랙을 접는다**" if k2v.startswith("❌")
       else "가르지 못했다(R33 ①)")))
CONFIG["K2"] = dict(err=K2, best_base=best_base, thr=float(K2_THR),
                    null=dict(mean=NR[0], lo=NR[1], hi=NR[2], n=int(NR[3])),
                    pi_hat={e: {str(r): PIH[e][r] for r in REC_OK} for e in PI_EST},
                    pi_true={str(r): BURD[r] for r in REC_OK})
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【K-C】 K3 적응 예산 · K4 오라클 상한 · K5 두 실패의 분리
run.log("\n" + "=" * 100)
run.log("【K-C】 K3(적응 예산) · K4(오라클 상한) · K5(저유병률 PPV vs 고유병률 민감도)")
run.log("=" * 100)
TOT = MAIN_K * NRE          # 총 예산은 균등 배분과 **같게** 유지한다

def alloc(w):
    """총 예산 TOT 을 가중 w 에 비례해 배분한다(최소 10개 보장)."""
    ww = np.array([max(1e-9, w[r]) for r in REC_OK], float); ww = ww / ww.sum()
    k = {r: int(max(10, round(TOT * ww[i]))) for i, r in enumerate(REC_OK)}
    return k

def eval_alloc(kmap):
    out = {}
    for r in REC_OK:
        out[r] = recall_at(r, kmap[r])
        out[r]["ach"] = out[r]["sens"] / out[r]["ceil"] if out[r]["ceil"] > 0 else np.nan
    return out

K_UNI = {r: MAIN_K for r in REC_OK}
best_e = NEW_EST if VERD["K2"].startswith("✅") else CONFIG["K2"]["best_base"]
K_ADA = alloc({r: PIH[best_e][r] * NN_[r] for r in REC_OK})

def greedy_oracle(total):
    """★★★ **환자 평균 민감도**를 겨냥한 탐욕 배분(비용 대비 이득).

    ⚠️ 함정 둘을 피해야 한다.
      ① `k ∝ π*` 는 **총 recall** 의 최적이지 **평균 민감도**의 최적이 **아니다** — 평균은
         레코드를 동등 가중하므로 S 가 작은 레코드의 한 칸이 훨씬 값지다(스모크 실측:
         `k ∝ π*` 가 균등보다 **나빴다** −0.1233).
      ② **예산 k 는 상위 k개를 통째로 가져간다.** 양성 칸만 골라 담으면 **실현 불가능한**
         배분이 된다 — 첫 판본이 그래서 균등보다 낮게 나왔다(0.3844 < 0.4161).
    그래서 「다음 양성까지 전진하는 비용」 대비 이득 `(1/S_r) / (j - k_r)` 이 가장 큰
    레코드에 예산을 준다. 접두사 제약을 지키므로 **실현 가능**하다."""
    import heapq
    nxt, cum, hp = {}, {}, []
    for r in REC_OK:
        idx = IDXS[r]; order = np.argsort(-L[idx])
        yy = TT_[idx][order].astype(bool)
        pos = np.where(yy)[0] + 1                     # 양성이 나오는 순위(1-based)
        nxt[r] = pos; cum[r] = 0                      # cum = 지금까지 준 예산
        if len(pos):
            heapq.heappush(hp, (-(1.0 / max(1, NS_[r])) / pos[0], r, 0, int(pos[0])))
    k = {r: 0 for r in REC_OK}; spent = 0
    while hp and spent < total:
        _, r, pi_, j = heapq.heappop(hp)
        cost = j - k[r]
        if cost <= 0:                                  # 이미 지나간 양성
            if pi_ + 1 < len(nxt[r]):
                jj = int(nxt[r][pi_ + 1])
                heapq.heappush(hp, (-(1.0 / max(1, NS_[r])) / max(1, jj - k[r]),
                                    r, pi_ + 1, jj))
            continue
        if spent + cost > total:
            continue
        k[r] = j; spent += cost
        if pi_ + 1 < len(nxt[r]):
            jj = int(nxt[r][pi_ + 1])
            heapq.heappush(hp, (-(1.0 / max(1, NS_[r])) / max(1, jj - k[r]),
                                r, pi_ + 1, jj))
    for r in REC_OK:                                   # 남은 예산은 균등하게 흘린다
        k[r] = max(1, k[r])
    left = total - sum(k.values())
    if left > 0:
        for i, r in enumerate(sorted(REC_OK, key=lambda x: NS_[x])):
            add = left // NRE + (1 if i < left % NRE else 0)
            k[r] += max(0, add)
    return k

K_ORA = greedy_oracle(TOT)
E_UNI, E_ADA, E_ORA = eval_alloc(K_UNI), eval_alloc(K_ADA), eval_alloc(K_ORA)

run.log(f"  총 예산 {TOT} 을 세 방식으로 배분 (적응 예산은 `{best_e}` 를 쓴다)")
run.log(f"  ⚠️ 오라클은 **`k ∝ π*` 가 아니라 탐욕 물채우기**다 — 평균 민감도는 레코드를")
run.log(f"     동등 가중하므로 `k ∝ π*`(총 recall 최적)가 **최적이 아니다**(스모크가 잡았다)")
run.log(f"  {'배분':<18}{'총 경보':>9}{'민감도 평균':>12}{'SD':>8}{'최소':>8}"
        f"{'PPV':>9}{'총 FN':>9}")
for nm, E, KM in (("균등(기록당 300)", E_UNI, K_UNI), (f"적응(∝ π̂)", E_ADA, K_ADA),
                  ("★ 오라클(탐욕) 상한", E_ORA, K_ORA)):
    sv = np.array([E[r]["sens"] for r in REC_OK]); pv = np.array([E[r]["ppv"] for r in REC_OK])
    run.log(f"  {nm:<18}{sum(KM.values()):>9d}{sv.mean():>12.4f}{sv.std(ddof=1):>8.4f}"
            f"{sv.min():>8.4f}{np.nanmean(pv):>9.4f}"
            f"{sum(E[r]['fn'] for r in REC_OK):>9d}")

ks = REC_OK
m3, lo3, hi3, n3 = boot_pair([E_UNI[r]["sens"] for r in ks], [E_ADA[r]["sens"] for r in ks],
                             SEED0 + 71, NB_BOOT)
m4, lo4, hi4, _ = boot_pair([E_UNI[r]["sens"] for r in ks], [E_ORA[r]["sens"] for r in ks],
                            SEED0 + 72, NB_BOOT)
run.log(f"\n    적응 − 균등  Δ민감도 **{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}]")
run.log(f"    오라클 − 균등 Δ민감도 **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}]  ← **상한**(R36 ①)")

# ── 영점 — π̂ 를 무작위로 섞어 배분하면?
# ★ 이 영점은 **배분만 바꾼다**(모델 재적합 없음) — 그래서 reps 를 크게 잡을 수 있다
run.log(f"\n  ★★ **영점** — 같은 π̂ 값 집합을 **레코드에 무작위 재배치**해 배분 "
        f"(reps={N_PERM_REG})")
nul3 = []
for s_ in range(N_PERM_REG):
    rr = np.random.RandomState(SEED0 + 600 + s_)
    perm = rr.permutation(NRE)
    w = {REC_OK[i]: PIH[best_e][REC_OK[perm[i]]] * NN_[REC_OK[i]] for i in range(NRE)}
    E_ = eval_alloc(alloc(w))
    nul3.append(float(np.mean([E_[r]["sens"] - E_UNI[r]["sens"] for r in ks])))
N3 = boot_mean(nul3, SEED0 + 62, NB_BOOT)
run.log(f"    영점 **{N3[0]:+.4f}** [{N3[1]:+.4f}, {N3[2]:+.4f}]")
K3_THR = max(0.0, N3[2]) if np.isfinite(N3[2]) else float("nan")
k3v = decide(lo3, hi3, K3_THR, ">")
g_("K3", k3v,
   f"적응 예산이 균등 대비 민감도 {m3:+.4f} [{lo3:+.4f}, {hi3:+.4f}] · 문턱 {K3_THR:+.4f} "
   f"(오라클 상한 {m4:+.4f}) — "
   + ("**부담 비례 배분이 실제로 민감도를 올린다**" if k3v.startswith("✅") else
      ("적응 배분이 오히려 해롭다" if k3v.startswith("❌") else "가르지 못했다(R33 ①)")))
REC_RATE = (m3 / m4) if (m4 > 1e-9 and m3 > 0) else float("nan")
g_("K4", "(관문 아님)",
   f"**오라클 예산(탐욕 물채우기)** Δ {m4:+.4f} [{lo4:+.4f}, {hi4:+.4f}] — 배분으로 얻을 수 "
   f"있는 **최대치**다(순위는 고정). "
   + (f"적응이 그 중 **{REC_RATE:.0%}** 를 회수했다" if np.isfinite(REC_RATE)
      else ("★ 상한이 0 이하라 **배분으로 얻을 게 없다** — 균등이 이미 최적에 가깝다"
            if m4 <= 1e-9 else
            "★ 적응이 **음수**라 회수율을 읽지 않는다 — π̂ 배분이 균등보다 해롭다")))

# ── K5 — 두 실패를 **따로** 본다
run.log(f"\n  ★★ K5 — **저유병률의 PPV 문제**와 **고유병률의 민감도 문제**는 다른 병이다")
qs = np.quantile(TRUE, [0, .25, .5, .75, 1.0])
run.log(f"  {'유병률 구간':<22}{'n':>4}{'상한':>8}{'민감도':>9}{'달성률':>9}{'PPV':>9}"
        f"{'오경보/기록':>12}")
K5 = []
for i in range(4):
    lo_, hi_ = qs[i], qs[i + 1]
    rs = [r for r in REC_OK if (lo_ <= BURD[r] <= hi_ if i == 3 else lo_ <= BURD[r] < hi_)]
    if not rs:
        continue
    row = dict(lo=float(lo_), hi=float(hi_), n=len(rs),
               ceil=float(np.mean([E_UNI[r]["ceil"] for r in rs])),
               sens=float(np.mean([E_UNI[r]["sens"] for r in rs])),
               ach=float(np.nanmean([E_UNI[r]["ach"] for r in rs])),
               ppv=float(np.nanmean([E_UNI[r]["ppv"] for r in rs])),
               fp=float(np.mean([E_UNI[r]["flagged"] - E_UNI[r]["tp"] for r in rs])))
    K5.append(row)
    run.log(f"  [{lo_:.4f}, {hi_:.4f}]{'':<6}{len(rs):>4}{row['ceil']:>8.3f}"
            f"{row['sens']:>9.4f}{row['ach']:>9.1%}{row['ppv']:>9.4f}{row['fp']:>12.0f}")
run.log(f"    ▸ 저유병률은 **상한 {K5[0]['ceil']:.2f} 로 여유**가 있는데 오경보가 "
        f"기록당 {K5[0]['fp']:.0f}개 — **PPV 문제**")
run.log(f"    ▸ 고유병률은 **상한 {K5[-1]['ceil']:.2f} 자체가 낮아** 민감도가 막힌다 — "
        f"**예산 문제**")
g_("K5", "(관문 아님)",
   f"저유병률 PPV {K5[0]['ppv']:.4f}(오경보 기록당 {K5[0]['fp']:.0f}개) vs 고유병률 상한 "
   f"{K5[-1]['ceil']:.3f}·민감도 {K5[-1]['sens']:.4f} — **하나의 예산으로 둘 다 못 만족한다**")
CONFIG["K3"] = dict(mean=m3, lo=lo3, hi=hi3, mde=float(mde(lo3, hi3)), thr=float(K3_THR),
                    null=dict(mean=N3[0], lo=N3[1], hi=N3[2]), est=best_e,
                    uni=dict(sens=float(np.mean([E_UNI[r]["sens"] for r in ks])),
                             fn=int(sum(E_UNI[r]["fn"] for r in REC_OK))),
                    ada=dict(sens=float(np.mean([E_ADA[r]["sens"] for r in ks])),
                             fn=int(sum(E_ADA[r]["fn"] for r in REC_OK))))
CONFIG["K4"] = dict(mean=m4, lo=lo4, hi=hi4,
                    ora=dict(sens=float(np.mean([E_ORA[r]["sens"] for r in ks])),
                             fn=int(sum(E_ORA[r]["fn"] for r in REC_OK))),
                    recovered=float(REC_RATE))
CONFIG["K5"] = K5
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【K-D】 필요표본 · ★ K6 검산표
run.log("\n" + "=" * 100)
run.log("【K-D】 필요표본 · K6 결론 검산표")
run.log("=" * 100)
run.log(f"  필요표본 (**관문 문턱 기준** · 레코드 · 현재 {NRE})")
for nm, m_, half_, thr_ in (("K2 ρ", K2[NEW_EST]["rho"],
                             mde(K2[NEW_EST]["rho_lo"], K2[NEW_EST]["rho_hi"]), K2_THR),
                            ("K3 민감도", CONFIG["K3"]["mean"], CONFIG["K3"]["mde"], K3_THR)):
    eff = m_ - thr_
    n5 = need_super(NRE, half_, eff); n8 = need_super(NRE, half_, eff, True)
    bad = (not np.isfinite(eff)) or abs(eff) < half_
    run.log(f"  {nm:<12}효과-문턱 {eff:>+8.4f} · 반폭 {half_:.4f} · n(50%) {n5:>6.0f} · "
            f"n(80%) {n8:>6.0f}  "
            + ("★ **해석 불가**(R41 ②)" if bad else
               ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

run.log("\n  ★ K6 — **결론 검산표**")
CHECK = [
    dict(claim=f"K1 — 예산 {MAIN_K}개에서 **상한에 걸린 레코드 "
               f"{CONFIG['K1']['n_capped']}/{NRE}**",
         num=f"상한 = min(1, k/S) · 달성률 평균 {CONFIG['K1']['ach_mean']:.4f} · "
             f"상한 중앙 {CONFIG['K1']['ceil_med']:.4f}",
         assume="**없음** — 상한은 예산과 양성 수만으로 정해진다",
         iffalse="★★★ Q4-F 의 J5 는 **예산 부족**(#48 상한 0.10)과 **모델 실패**"
                 "(#38 상한 0.67·달성 1.3%)를 **한 표에 섞었다**. 그게 이 런의 정정이다"),
    dict(claim=f"K1 — ρ(AUROC, 달성률) = {CONFIG['K1']['rho_auc_ach']:+.4f} · "
               f"AUROC>0.85 인데 달성률<50% 인 레코드 {CONFIG['K1']['n_hi_auc_lo_ach']}개",
         num=f"ρ(lift, 달성률) = {CONFIG['K1']['rho_lift_ach']:+.4f} — PR-AUC lift 가 "
             f"AUROC 보다 상위 꼬리를 잘 대변한다",
         assume="**없음** — 같은 점수에서 셋을 냈다",
         iffalse="★★★ **AUROC 는 전체 순위를 보고 예산 국면은 최상위 꼬리만 쓴다**(R40 ①). "
                 "Q4-E 의 「AUROC 0.9418」은 참이지만 **배포 성능을 대변하지 않는다**"),
    dict(claim=f"K2 — 기록 수준 회귀 ρ {K2[NEW_EST]['rho']:+.4f} "
               f"[{K2[NEW_EST]['rho_lo']:+.4f}, {K2[NEW_EST]['rho_hi']:+.4f}] → {VERD['K2']}",
         num=f"기존 최선 `{CONFIG['K2']['best_base']}` ρ "
             f"{K2[CONFIG['K2']['best_base']]['rho']:+.4f} · 영점 "
             f"{CONFIG['K2']['null']['mean']:+.4f} · 문턱 {K2_THR:+.4f} · "
             f"특징 {NFEAT}개 · ridge α={RIDGE_A}",
         assume="회귀를 **held-out 을 뺀 레코드에서만** 적합했다(R22)",
         iffalse="★★★ 기존 넷이 ρ≈0 인 건 **전부 박동별 보정 확률의 함수**여서 확률 눈금에 "
                 "갇히기 때문이다. 회귀가 그 경로를 우회한다"),
    dict(claim=f"K3 — 적응 예산 Δ민감도 {CONFIG['K3']['mean']:+.4f} → {VERD['K3']}",
         num=f"영점 {CONFIG['K3']['null']['mean']:+.4f}(같은 π̂ 값 집합을 **무작위 재배치**) · "
             f"오라클 상한(탐욕) {CONFIG['K4']['mean']:+.4f} · 회수율 "
             + (f"{CONFIG['K4']['recovered']:.0%}"
                if np.isfinite(CONFIG['K4']['recovered']) else "(상한 ≤ 0 — 안 읽는다)"),
         assume="**총 예산을 균등 배분과 같게** 유지했다",
         iffalse="★ 총 예산을 안 맞추면 더 많이 울려서 이길 수 있다(R36 ②)"),
    dict(claim=f"K5 — 저유병률 PPV {CONFIG['K5'][0]['ppv']:.4f} vs 고유병률 상한 "
               f"{CONFIG['K5'][-1]['ceil']:.3f}",
         num=f"저유병률은 오경보가 기록당 {CONFIG['K5'][0]['fp']:.0f}개 · 고유병률은 상한이 "
             f"막는다",
         assume="**없음** — 같은 예산에서 두 사분위를 따로 냈다",
         iffalse="★★ **하나의 예산으로 둘 다 못 만족한다** — 그래서 적응 예산이 필요하고, "
                 "적응 예산은 π̂ 가 필요하다(K2 가 그 전제다)"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["K6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【K-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

pv = np.array([BURD[r] for r in REC_OK])
ax[0].scatter(pv, [R1[r]["ceil"] for r in REC_OK], s=30, color="tab:red", label="ceiling")
ax[0].scatter(pv, [R1[r]["sens"] for r in REC_OK], s=30, color="tab:blue", marker="^",
              label="sensitivity")
ax[0].set_xscale("log"); ax[0].set_xlabel("record prevalence (log)")
ax[0].set_ylabel(f"@ {MAIN_K}/record")
ax[0].set_title("K1 : the ceiling, which Q4-F did not report", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

xs = np.arange(len(PI_EST))
cols = ["tab:gray"] * len(PI_EST); cols[PI_EST.index(NEW_EST)] = "tab:green"
ax[1].bar(xs, [K2[e]["rho"] for e in PI_EST], color=cols)
ax[1].axhline(0, color="k", lw=.9)
ax[1].axhline(K2_THR, ls=":", color="tab:red", lw=1.2, label=f"thr {K2_THR:+.3f}")
ax[1].set_xticks(xs); ax[1].set_xticklabels(PI_EST, fontsize=8)
ax[1].set_ylabel("rho(pi_hat, pi_true)")
ax[1].set_title("K2 : can burden be read at all", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="y")

ax[2].scatter(pv, [PIH[CONFIG["K2"]["best_base"]][r] for r in REC_OK], s=26,
              color="tab:gray", label=CONFIG["K2"]["best_base"])
ax[2].scatter(pv, [PIH[NEW_EST][r] for r in REC_OK], s=30, color="tab:green",
              marker="^", label=NEW_EST)
ax[2].plot([0, pv.max()], [0, pv.max()], "k--", lw=1.0)
ax[2].set_xlabel("true burden"); ax[2].set_ylabel("estimated burden")
ax[2].set_title("K2 : record-level regression vs beat-level", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4g_budget_ceiling_burden", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **K1 눈금 정정** — 예산 {MAIN_K}개에서 상한에 걸린 레코드 "
        f"**{CONFIG['K1']['n_capped']}/{NRE}** · 달성률 평균 {CONFIG['K1']['ach_mean']:.4f}")
run.log(f"     ρ(AUROC, 달성률) **{CONFIG['K1']['rho_auc_ach']:+.4f}** · "
        f"ρ(lift, 달성률) {CONFIG['K1']['rho_lift_ach']:+.4f}")
run.log(f"     → **AUROC 는 배포 성능을 대변하지 않는다** — Q4-E 의 0.9418 은 참이지만 "
        f"예산 국면은 **최상위 꼬리**만 쓴다")
run.log("")
run.log(f"  ★★★ **K2 부담 추정** — 기록 수준 회귀 ρ {K2[NEW_EST]['rho']:+.4f} "
        f"[{K2[NEW_EST]['rho_lo']:+.4f}, {K2[NEW_EST]['rho_hi']:+.4f}] · MAE "
        f"{K2[NEW_EST]['mae']:.4f}")
run.log(f"     기존 최선 `{CONFIG['K2']['best_base']}` ρ "
        f"{K2[CONFIG['K2']['best_base']]['rho']:+.4f} · MAE "
        f"{K2[CONFIG['K2']['best_base']]['mae']:.4f} → {VERD['K2']}")
if ok_("K2"):
    run.log("     → ★★★ **부담이 기록 수준에서 읽힌다. 정량 트랙이 열린다.**")
elif VERD["K2"].startswith("❌"):
    run.log("     → ⛔ **기록 수준으로도 못 읽는다 — 정량 트랙을 접는다**(상한만 남긴다)")
run.log("")
run.log(f"  ★★ **K3 적응 예산** Δ민감도 {CONFIG['K3']['mean']:+.4f} "
        f"[{CONFIG['K3']['lo']:+.4f}, {CONFIG['K3']['hi']:+.4f}] → {VERD['K3']}")
run.log(f"     오라클 상한 {CONFIG['K4']['mean']:+.4f} · 회수율 "
        + (f"{CONFIG['K4']['recovered']:.0%}" if np.isfinite(CONFIG['K4']['recovered'])
           else "(상한 ≤ 0 — 읽지 않는다)")
        + f" · FN {CONFIG['K3']['uni']['fn']} → "
        f"{CONFIG['K3']['ada']['fn']}(오라클 {CONFIG['K4']['ora']['fn']})")
run.log("")
run.log(f"  ★ **K5 두 병이 다르다** — 저유병률 PPV {CONFIG['K5'][0]['ppv']:.4f}"
        f"(오경보 기록당 {CONFIG['K5'][0]['fp']:.0f}개) vs 고유병률 상한 "
        f"{CONFIG['K5'][-1]['ceil']:.3f}·민감도 {CONFIG['K5'][-1]['sens']:.4f}")

run.finish({
    "exp_id": "quest46_q4g_budget_ceiling_burden",
    "metric": "record_level_burden_rho",
    "value": float(K2[NEW_EST]["rho"]),
    "passed": bool(ok_("K0") and ok_("K2")),
    "summary": ("Q4-F 의 지표 오류를 고치고 막힌 자리를 새 접근으로 뚫었다. K1 이 **예산 "
                "상한**(min(1, k/S))과 **달성률**을 도입해 Q4-F J5 가 섞어 놓은 두 실패"
                "(예산 부족 vs 모델 실패)를 갈랐고, **recall@k 와 AUROC 의 괴리**를 실측해 "
                "AUROC 가 예산 국면의 요약이 아님을 보였다. K2 는 기존 네 추정기가 ρ≈0 인 "
                "이유(전부 박동별 보정 확률의 함수라 확률 눈금에 갇힌다)를 우회해 **기록 "
                "수준 요약통계에서 π 를 직접 회귀**했다. K3·K4 가 적응 예산과 그 오라클 "
                "상한을 재고, K5 가 저유병률의 PPV 문제와 고유병률의 예산 문제를 분리했다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "K0": CONFIG.get("K0", {}),
    "K1": CONFIG.get("K1", {}), "K2": CONFIG.get("K2", {}), "K3": CONFIG.get("K3", {}),
    "K4": CONFIG.get("K4", {}), "K5": CONFIG.get("K5", []),
    "K6": CONFIG.get("K6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4g_budget_ceiling_burden.ipynb`")
